In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# 1. Setup Configurations
np.random.seed(42)
total_trucks = 1300
records_per_truck = 15  
total_records = total_trucks * records_per_truck
start_date = datetime(2026, 1, 1)

# Refined Products Depot and Destination Corridors Map
depot_routes = {
    "Apapa Port Refinery Depot": {"Route": "Lagos-Ibadan Expressway Corridor", "Base_KM": 130, "Base_Hours": 2.5},
    "Port Harcourt Refining Complex": {"Route": "Port Harcourt-Aba Road Corridor", "Base_KM": 65, "Base_Hours": 1.8},
    "Warri Delta Depot": {"Route": "Sagamu-Benin Express Corridor", "Base_KM": 260, "Base_Hours": 5.0}
}

truck_ids = [f"OG-FLEET-{1000 + i}" for i in range(total_trucks)]
truck_status_pool = (["In Transit"] * 850) + (["Idle at Depot"] * 250) + (["Under Maintenance"] * 120) + (["Off-Duty"] * 80)

data = []

# 2. Simulation Loop
for truck in truck_ids:
    current_status = random.choice(truck_status_pool)
    hours_since_service = random.randint(10, 450)
    
    # --- SMART MAINTENANCE DISPATCH LOGIC ---
    if hours_since_service >= 310:
        maintenance_recommendation = "Maintenance Warning: Restrict to Short Routes"
        allowed_depots = ["Apapa Port Refinery Depot", "Port Harcourt Refining Complex"]
        origin_depot = random.choice(allowed_depots)
    else:
        maintenance_recommendation = "Optimal Operations"
        origin_depot = random.choice(list(depot_routes.keys()))
        
    route_info = depot_routes[origin_depot]
    base_distance = route_info["Base_KM"]
    base_time = route_info["Base_Hours"]
    
    for log_idx in range(records_per_truck):
        timestamp = start_date + timedelta(hours=log_idx * 4)
        fuel_siphoned = 0.0
        fuel_theft_flag = 0
        
        if current_status == "In Transit":
            # Mid-Transit Siphoning Logic
            if random.random() > 0.95:  
                speed = 0 
                distance_covered = round(random.uniform(10, base_distance - 5), 1)
                normal_fuel_consumed = round(distance_covered * 0.4, 1)
                fuel_siphoned = round(random.uniform(50.0, 150.0), 1) 
                fuel_theft_flag = 1
                log_status = "In Transit (Stopped/Unscheduled)"
            else:
                speed = random.randint(45, 85)
                distance_covered = round(random.uniform(10, base_distance), 1)
                normal_fuel_consumed = round(distance_covered * 0.4, 1)
                log_status = "In Transit"
        else:
            speed = 0
            distance_covered = 0.0
            normal_fuel_consumed = random.uniform(0.2, 1.5)
            log_status = current_status
            
            # Yard Siphoning Logic
            if current_status in ["Idle at Depot", "Off-Duty"] and random.random() > 0.96:
                fuel_siphoned = round(random.uniform(40.0, 120.0), 1)
                fuel_theft_flag = 1

        total_fuel_loss = round(normal_fuel_consumed + fuel_siphoned, 1)
        engine_temp = np.random.normal(92, 7) if speed > 0 else np.random.normal(45, 5)
        vibration = np.random.normal(2.2, 0.6) if speed > 0 else np.random.normal(0.2, 0.1)
        
        # Breakdown Risk Injection Framework
        breakdown_target = 0
        if hours_since_service > 350 or engine_temp > 110:
            if speed > 0 and random.random() > 0.4:
                breakdown_target = 1
                vibration += 3.5

        hour_of_day = timestamp.hour
        traffic_delay = random.uniform(1.0, 4.0) if (6 <= hour_of_day <= 9 or 16 <= hour_of_day <= 20) else random.uniform(0.1, 0.8)
        actual_travel_time = base_time + traffic_delay + np.random.normal(0.2, 0.05) if speed > 0 else 0.0

        data.append([
            timestamp, truck, log_status, origin_depot, route_info["Route"], 
            base_distance, distance_covered, speed, total_fuel_loss, 
            fuel_theft_flag, round(engine_temp, 1), hours_since_service, 
            maintenance_recommendation, round(vibration, 2), breakdown_target, round(actual_travel_time, 2)
        ])

# 3. Compile Schema & Save
columns = [
    "Timestamp", "Truck_ID", "Fleet_Status", "Origin_Depot", "Assigned_Route",
    "Total_Route_Distance_KM", "Distance_Covered_KM", "Current_Speed_KMH", "Fuel_Volume_Lost_Liters",
    "Fuel_Theft_Anomaly", "Engine_Temp_C", "Hours_Since_Service", "Maintenance_Recommendation", 
    "Vibration_G", "Breakdown_Risk_Target", "Actual_Trip_Duration_Hours"
]
df = pd.DataFrame(data, columns=columns)
df.to_csv("fleet_telematics_database.csv", index=False)
print(f"Data pipeline complete. Generated {df.shape} tracking logs incorporating smart scheduling.")


Data pipeline complete. Generated (19500, 16) tracking logs incorporating smart scheduling.
